# Automated Review Reliability Analysis with the OpenAI API

This notebook implements an LLM-based pipeline to analyse Amazon product reviews for three tasks:

1. **Sentiment classification**  
   Identifying whether the overall opinion expressed in a review is positive, negative, or neutral.

2. **Rating–Text inconsistency detection**  
   Detecting mismatches between the numerical star rating and the sentiment conveyed in the review text.

3. **Evaluation of Model Performance**  
   Assessing the effectiveness of the approach using both automated metrics and LLM-as-judge evaluation.

## Notebook Structure

This notebook is organised into the following stages:

1. Setup and imports  
2. Data loading and preprocessing  
3. Balanced sampling and rule-based baseline  
4. Prompt design  
5. Helper functions for API interaction and JSON parsing  
6. Prompt execution  
7. Quantitative evaluation  
8. LLM-as-judge evaluation  
9. Examples of model success and failure 
10. Final summary

## 1. Setup and Imports

This section imports the required Python libraries, defines model configuration, and initializes the OpenAI client.

In [1]:
# Standard libraries
import os
import json
import time
import re
# Data handling
import pandas as pd
# Metrics
from sklearn.metrics import accuracy_score

# OpenAI client
from openai import OpenAI

In [ ]:
# Temporary key setup
API_KEY = "API_key_here"
os.environ["OPENAI_API_KEY"] = API_KEY

# Models used in the project
MODEL_NAME = "gpt-4o-mini"
JUDGE_MODEL_NAME = "gpt-4o-mini"

# OpenAI client
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])


## 2. Data Loading and Preprocessing

The Amazon reviews dataset (10000 sample with balanced sampling strategy) is loaded and filtered to retain only the fields needed for this project.  
The preprocessing steps also create derived variables used later in the analysis.

In [2]:

# Load the dataset (10,000 reviews for testing from the original ~500,000)
DATA_PATH = "sample_reviews_10000.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully.")
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

df.head()


Dataset loaded successfully.
Shape: (10000, 10)

Columns:
['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,98039,B0002MLA5K,A1PZ7S1L36QRKJ,Marcella Y Giffen,0,2,1,1340928000,Close call!,I took my 10 year old cat in to the vet today ...
1,373810,B004WTHCO2,A1CXB89TV5KW3U,abmama,3,5,1,1297641600,"buy a baby brezza instead, save money and time...",from reading all the reviews about black gunk ...
2,60696,B003QNJYXM,A30UZHA2QAVSMY,JP,6,13,1,1253577600,Placebo,My girlfriend and I decided to try 5-hour ener...
3,319567,B0007LXU9A,AF4171OIS1TC7,"R. Singavarapu ""virgodoc""",1,6,1,1178928000,Great stuff but bad quality,Looks like Amazon is selling old stuff. Kashi ...
4,188734,B000HDONP8,A2HOA352TOCY91,Grace,3,4,1,1274054400,Tasted too grainy,Didn't like it that it felt like I was eating ...


### 2.1 Select Relevant Columns and Clean Text

The notebook keeps only the variables required for review reliability analysis. Missing values are removed and the title and body are combined into a single review text field.

In [4]:
# Keep only the columns needed for the project
required_columns = [
    "Id",
    "ProductId",
    "HelpfulnessNumerator",
    "HelpfulnessDenominator",
    "Score",
    "Summary",
    "Text",
]

df = df[required_columns].copy()
# Remove rows with missing rating/title/text
df = df.dropna(subset=["Score", "Summary", "Text"]).copy()
# Standardize text fields
df["Score"] = df["Score"].astype(int)
df["Summary"] = df["Summary"].astype(str).str.strip()
df["Text"] = df["Text"].astype(str).str.strip()
# Combine title and body into one analysis field
df["review_text"] = (df["Summary"] + ". " + df["Text"]).str.strip()

df = df[df["review_text"] != "."].copy()

print("Cleaned dataset shape:", df.shape)
df[["Id", "Score", "Summary", "Text", "review_text"]].head()


Cleaned dataset shape: (9999, 8)


,Id,Score,Summary,Text,review_text
0,98039,1,Close call!,I took my 10 year old cat in to the vet today ...,Close call!. I took my 10 year old cat in to t...
1,373810,1,"buy a baby brezza instead, save money and time...",from reading all the reviews about black gunk ...,"buy a baby brezza instead, save money and time..."
2,60696,1,Placebo,My girlfriend and I decided to try 5-hour ener...,Placebo. My girlfriend and I decided to try 5-...
3,319567,1,Great stuff but bad quality,Looks like Amazon is selling old stuff. Kashi ...,Great stuff but bad quality. Looks like Amazon...
4,188734,1,Tasted too grainy,Didn't like it that it felt like I was eating ...,Tasted too grainy. Didn't like it that it felt...


### 2.2 Feature Engineering

Additional variables are created to support later evaluation:
- `review_text`: combined title and body
- `helpfulness_ratio`: helpfulness numerator divided by denominator
- `has_helpfulness_vote`: whether the review has any helpfulness votes

In [5]:
# Helper function to calculate helpfulness ratio safely
def safe_helpfulness_ratio(num, den):
    if pd.isna(den) or den == 0:
        return 0.0
    return num / den
# Add derived features
df["helpfulness_ratio"] = df.apply(
    lambda row: safe_helpfulness_ratio(row["HelpfulnessNumerator"], row["HelpfulnessDenominator"]),
    axis=1,
)

print("Score distribution:")
print(df["Score"].value_counts().sort_index())


Score distribution:
Score
1    2000
2    1999
3    2000
4    2000
5    2000
Name: count, dtype: int64


## 3. Balanced Sampling and Baseline Construction

To avoid strong rating imbalance in the full dataset, a balanced sample is drawn across review scores.  
This makes prompt comparison easier and allows a fairer small-scale evaluation.

In [6]:
# Sample a balanced subset of the data for evaluation
SAMPLE_PER_SCORE = 20
RANDOM_STATE = 16

sampled_groups = []
for score, group in df.groupby("Score"):
    n = min(SAMPLE_PER_SCORE, len(group))
    sampled_groups.append(group.sample(n=n, random_state=RANDOM_STATE))

df_sample = pd.concat(sampled_groups).reset_index(drop=True)

print("Sample size:", len(df_sample))
print(df_sample["Score"].value_counts().sort_index())


Sample size: 100
Score
1    20
2    20
3    20
4    20
5    20
Name: count, dtype: int64


### 3.1 Score-Derived Expected Sentiment

A simple expected sentiment label is derived from the star rating:
- 1-2 stars: negative
- 3 stars: neutral
- 4-5 stars: positive

These labels are used as a reference point for evaluation.

In [7]:
# Define expected sentiment based on score
def expected_sentiment_from_score(score):
    if score in [1, 2]:
        return "negative"
    if score == 3:
        return "neutral"
    if score in [4, 5]:
        return "positive"
    return None

# Add expected sentiment column to the sample
df_sample["expected_sentiment"] = df_sample["Score"].apply(expected_sentiment_from_score)


### 3.2 Rule-Based Baseline

A lightweight keyword-based baseline is created to provide a non-LLM comparison.  
This baseline classifies sentiment using simple positive and negative word lists.

In [8]:
# Define simple baseline sentiment analysis using keyword matching
positive_words = {
    "good", "great", "excellent", "amazing", "love", "best", "wonderful",
    "perfect", "nice", "delicious", "happy", "satisfied", "fantastic",
    "awesome", "recommend", "fresh", "enjoy", "enjoyed", "soft",
} # This is a very basic list and can be expanded significantly for better performance.

negative_words = {
    "bad", "terrible", "awful", "worst", "hate", "poor", "disappointing",
    "disappointed", "broken", "stale", "hard", "waste", "horrible",
    "problem", "problems", "refund", "return", "not", "never",
} # This is a very basic list and can be expanded significantly for better performance.

# Simple baseline sentiment analysis function
def simple_baseline_sentiment(text):
    text = str(text).lower()
    tokens = re.findall(r"\b[a-z]+\b", text)
    pos_count = sum(token in positive_words for token in tokens)
    neg_count = sum(token in negative_words for token in tokens)
    if pos_count > neg_count:
        return "positive"
    if neg_count > pos_count:
        return "negative"
    return "neutral"

# Apply baseline sentiment analysis and compare with expected sentiment
df_sample["baseline_sentiment"] = df_sample["review_text"].apply(simple_baseline_sentiment)
df_sample["baseline_inconsistency"] = df_sample["baseline_sentiment"] != df_sample["expected_sentiment"]


### 3.3 Helpfulness-based Quality Proxy

A proxy label for review quality is derived from helpfulness votes. This is not a gold-standard annotation, but it provides a rough reference for comparing LLM quality predictions.

In [9]:
# Define a quality proxy based on helpfulness votes
def helpfulness_quality_proxy(row):
    den = row["HelpfulnessDenominator"]
    ratio = row["helpfulness_ratio"]
    if den == 0:
        return "unknown"
    if den >= 3 and ratio >= 0.60:
        return "high"
    if den >= 3 and ratio < 0.40:
        return "low"
    return "medium"

# Add quality proxy column to the sample
df_sample["quality_proxy"] = df_sample.apply(helpfulness_quality_proxy, axis=1)

# Display the sample with relevant columns
df_sample[
    ["Id", "Score", "expected_sentiment", "baseline_sentiment", "baseline_inconsistency", "quality_proxy"]
].head()

,Id,Score,expected_sentiment,baseline_sentiment,baseline_inconsistency,quality_proxy
0,411540,1,negative,neutral,True,medium
1,54080,1,negative,negative,False,unknown
2,214242,1,negative,negative,False,unknown
3,89016,1,negative,negative,False,high
4,505107,1,negative,negative,False,low


## 4. Prompt Design

Two prompt variants are tested in order to examine how prompt structure affects output quality.

- **Prompt A**: shorter and more direct
- **Prompt B**: more explicit definitions and guidance

Both prompts request strict JSON output.

In [10]:
PROMPT_A = """
You are a review reliability analysis assistant.

Analyze only the review in the Input section.

Tasks:
1. Determine sentiment: positive, negative, or neutral
2. Detect inconsistency between star rating and review text: true or false
3. Assess review quality: high, medium, or low
4. Provide short reasoning in no more than 40 words

Return strict JSON only with no markdown and no extra text.

Output format:
{
  "sentiment": "positive | negative | neutral",
  "inconsistency": true,
  "review_quality": "high | medium | low",
  "reasoning": "short explanation"
}
"""


In [11]:
PROMPT_B = """
You are an expert evaluator of online review reliability.

Analyze only the review in the Input section.

Definitions:
- Sentiment = overall opinion expressed in the text: positive, negative, or neutral
- Inconsistency = true when the review text sentiment does not match the expected sentiment implied by the star rating
  * 1-2 stars usually imply negative sentiment
  * 3 stars usually imply neutral or mixed sentiment
  * 4-5 stars usually imply positive sentiment
- Review quality:
  * high = specific, informative, and supported by details
  * medium = somewhat informative but limited in detail
  * low = very short, vague, generic, or uninformative

Instructions:
- Use only the provided star rating and review text
- Do not infer extra facts beyond the input
- Keep reasoning under 40 words
- Return strict JSON only with no markdown and no extra text

Output format:
{
  "sentiment": "positive | negative | neutral",
  "inconsistency": true,
  "review_quality": "high | medium | low",
  "reasoning": "short explanation"
}
"""


## 5. Helper Functions for Robust API Use

This section defines utility functions for:
- extracting text from model responses
- parsing JSON safely
- handling malformed outputs

In [12]:

# Function to extract response text safely
def get_response_text(response):
    return getattr(response, "output_text", "").strip()

# Function to safely parse JSON from model response
def safe_json_parse(text, required_keys=None):
    required_keys = required_keys or []

    if text is None:
        return {}, True

    text = text.strip()

    # Remove markdown code fences if present
    if text.startswith("```json"):
        text = text[7:]
    elif text.startswith("```"):
        text = text[3:]

    if text.endswith("```"):
        text = text[:-3]

    text = text.strip()

    # Extract JSON object only
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        text = text[start:end + 1]
    else:
        return {}, True

    try:
        parsed = json.loads(text)

        if not isinstance(parsed, dict):
            return {}, True

        missing_keys = [k for k in required_keys if k not in parsed]
        if missing_keys:
            return parsed, True

        return parsed, False

    except Exception:
        return {}, True

### 5.1 Review Analysis Function

The core function sends one review at a time to the OpenAI API and returns:
- raw model output
- parsed sentiment
- parsed inconsistency flag
- parsed review quality
- reasoning
- parse and API error indicators

In [13]:
# Main function to analyze a review with retries and error handling
def analyze_review_with_llm(row, system_prompt, model_name=MODEL_NAME, max_retries=2, pause_seconds=0.0):
    review_input = f"""
Input:
Star Rating: {row['Score']}
Review Title: {row['Summary']}
Review Text: {row['Text']}
""".strip()

    required_keys = ["sentiment", "inconsistency", "review_quality", "reasoning"]
    last_error = None

    for attempt in range(max_retries + 1):
        try:
            response = client.responses.create(
                model=model_name,
                input=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": review_input},
                ],
                temperature=0,
                max_output_tokens=120,
            )

            raw_output = get_response_text(response)
            parsed_output, parse_error = safe_json_parse(raw_output, required_keys=required_keys)

            result = {
                "raw_output": raw_output,
                "llm_sentiment": parsed_output.get("sentiment"),
                "llm_inconsistency": parsed_output.get("inconsistency"),
                "llm_review_quality": parsed_output.get("review_quality"),
                "llm_reasoning": parsed_output.get("reasoning"),
                "parse_error": parse_error,
                "api_error": False,
                "api_error_message": None,
            }
            time.sleep(pause_seconds)
            return result

        except Exception as e:
            last_error = str(e)
            time.sleep(min(2 ** attempt, 4))

    return {
        "raw_output": None,
        "llm_sentiment": None,
        "llm_inconsistency": None,
        "llm_review_quality": None,
        "llm_reasoning": None,
        "parse_error": True,
        "api_error": True,
        "api_error_message": last_error,
    }


## 6. Run Prompt Experiments

Both Prompt A and Prompt B are applied to the same balanced sample so that their outputs can be compared directly.

In [14]:
results_a = []
# Analyze reviews with the first prompt and collect results
for _, row in df_sample.iterrows():
    llm_result = analyze_review_with_llm(row, PROMPT_A)

    combined = {
        "Id": row["Id"],
        "ProductId": row["ProductId"],
        "Score": row["Score"],
        "Summary": row["Summary"],
        "Text": row["Text"],
        "review_text": row["review_text"],
        "expected_sentiment": row["expected_sentiment"],
        "baseline_sentiment": row["baseline_sentiment"],
        "baseline_inconsistency": row["baseline_inconsistency"],
        "helpfulness_ratio": row["helpfulness_ratio"],
        "quality_proxy": row["quality_proxy"],
        "prompt_a_sentiment": llm_result["llm_sentiment"],
        "prompt_a_inconsistency": llm_result["llm_inconsistency"],
        "prompt_a_review_quality": llm_result["llm_review_quality"],
        "prompt_a_reasoning": llm_result["llm_reasoning"],
        "prompt_a_raw_output": llm_result["raw_output"],
        "prompt_a_parse_error": llm_result["parse_error"],
        "prompt_a_api_error": llm_result["api_error"],
        "prompt_a_api_error_message": llm_result["api_error_message"],
    }
    results_a.append(combined)

results_a_df = pd.DataFrame(results_a)
print("Prompt A run complete:", results_a_df.shape)
results_a_df.head()


Prompt A run complete: (100, 19)


,Id,ProductId,Score,Summary,Text,review_text,expected_sentiment,baseline_sentiment,baseline_inconsistency,helpfulness_ratio,quality_proxy,prompt_a_sentiment,prompt_a_inconsistency,prompt_a_review_quality,prompt_a_reasoning,prompt_a_raw_output,prompt_a_parse_error,prompt_a_api_error,prompt_a_api_error_message
0,411540,B00382U0VG,1,These are disgusting!,Biting into one of these is like trying to eat...,These are disgusting!. Biting into one of thes...,negative,neutral,True,0.000000,medium,negative,False,high,The review clearly expresses strong dissatisfa...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
1,54080,B000LKVAYO,1,awful,i wish there was a way to give zero stars. thi...,awful. i wish there was a way to give zero sta...,negative,negative,False,0.000000,unknown,negative,False,high,The review clearly expresses strong dissatisfa...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
2,214242,B00061MUR4,1,poor quality,this is a waste of money. in my opinion the bo...,poor quality. this is a waste of money. in my ...,negative,negative,False,0.000000,unknown,negative,False,high,The review clearly expresses dissatisfaction a...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
3,89016,B000W642B0,1,Looks bigger than actual size!,I received this as a gift. Company enclosed a...,Looks bigger than actual size!. I received thi...,negative,negative,False,0.750000,high,negative,False,high,The review clearly expresses dissatisfaction w...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
4,505107,B000LKTXIO,1,Worst Bread Ever!!!,My husband bought this bread for us to try and...,Worst Bread Ever!!!. My husband bought this br...,negative,negative,False,0.111111,low,negative,False,high,The review clearly expresses dissatisfaction w...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None


In [15]:
results_b = []
# Analyze the same reviews with the second prompt for comparison
for _, row in df_sample.iterrows():
    llm_result = analyze_review_with_llm(row, PROMPT_B)

    combined = {
        "Id": row["Id"],
        "prompt_b_sentiment": llm_result["llm_sentiment"],
        "prompt_b_inconsistency": llm_result["llm_inconsistency"],
        "prompt_b_review_quality": llm_result["llm_review_quality"],
        "prompt_b_reasoning": llm_result["llm_reasoning"],
        "prompt_b_raw_output": llm_result["raw_output"],
        "prompt_b_parse_error": llm_result["parse_error"],
        "prompt_b_api_error": llm_result["api_error"],
        "prompt_b_api_error_message": llm_result["api_error_message"],
    }
    results_b.append(combined)

results_b_df = pd.DataFrame(results_b)
print("Prompt B run complete:", results_b_df.shape)
results_b_df.head()


Prompt B run complete: (100, 9)


,Id,prompt_b_sentiment,prompt_b_inconsistency,prompt_b_review_quality,prompt_b_reasoning,prompt_b_raw_output,prompt_b_parse_error,prompt_b_api_error,prompt_b_api_error_message
0,411540,negative,False,high,The review provides specific details about the...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
1,54080,negative,False,high,The review clearly expresses strong negative f...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
2,214242,negative,False,high,The review clearly expresses dissatisfaction w...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
3,89016,negative,False,high,The review expresses clear dissatisfaction wit...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
4,505107,negative,False,high,The review clearly expresses dissatisfaction w...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None




The results from both prompts are merged into a single dataframe for downstream evaluation and comparison.

In [16]:
# Merge results from both prompts on Id
results_df = results_a_df.merge(results_b_df, on="Id", how="left")
print("Merged results shape:", results_df.shape)
results_df.head()


Merged results shape: (100, 27)


,Id,ProductId,Score,Summary,Text,review_text,expected_sentiment,baseline_sentiment,baseline_inconsistency,helpfulness_ratio,...,prompt_a_api_error,prompt_a_api_error_message,prompt_b_sentiment,prompt_b_inconsistency,prompt_b_review_quality,prompt_b_reasoning,prompt_b_raw_output,prompt_b_parse_error,prompt_b_api_error,prompt_b_api_error_message
0,411540,B00382U0VG,1,These are disgusting!,Biting into one of these is like trying to eat...,These are disgusting!. Biting into one of thes...,negative,neutral,True,0.000000,...,False,None,negative,False,high,The review provides specific details about the...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
1,54080,B000LKVAYO,1,awful,i wish there was a way to give zero stars. thi...,awful. i wish there was a way to give zero sta...,negative,negative,False,0.000000,...,False,None,negative,False,high,The review clearly expresses strong negative f...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
2,214242,B00061MUR4,1,poor quality,this is a waste of money. in my opinion the bo...,poor quality. this is a waste of money. in my ...,negative,negative,False,0.000000,...,False,None,negative,False,high,The review clearly expresses dissatisfaction w...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
3,89016,B000W642B0,1,Looks bigger than actual size!,I received this as a gift. Company enclosed a...,Looks bigger than actual size!. I received thi...,negative,negative,False,0.750000,...,False,None,negative,False,high,The review expresses clear dissatisfaction wit...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None
4,505107,B000LKTXIO,1,Worst Bread Ever!!!,My husband bought this bread for us to try and...,Worst Bread Ever!!!. My husband bought this br...,negative,negative,False,0.111111,...,False,None,negative,False,high,The review clearly expresses dissatisfaction w...,"{\n ""sentiment"": ""negative"",\n ""inconsistenc...",False,False,None


## 7. Quantitative Evaluation

This section evaluates model outputs through automated comparisons across multiple dimensions:

- **Sentiment agreement**: the proportion of cases where the predicted sentiment matches the sentiment derived from the star rating  
- **Inconsistency agreement**: the extent to which the model’s inconsistency prediction aligns with a logically derived inconsistency  
- **Parse error rate**: the proportion of outputs that fail to conform to the required JSON schema  
- **Inconsistency rate (rating-derived baseline)**: the frequency of mismatches when sentiment is inferred directly from star ratings 

### Inconsistency Derivation

To evaluate logical consistency, a derived inconsistency label is constructed using two variables: the model’s predicted sentiment (predicted_sentiment) and the rating-based expected sentiment (expected_sentiment).

Specifically, inconsistency is defined as a mismatch between these two variables:

- derived_inconsistency = True, if predicted_sentiment ≠ expected_sentiment  
- derived_inconsistency = False, otherwise  

This derived label serves as a proxy ground truth, as the dataset does not provide explicit annotations for inconsistency. It is later used to evaluate the model’s predicted inconsistency.

In [17]:
# Derive inconsistency by comparing predicted_sentiment with expected_sentiment
def derive_inconsistency(predicted_sentiment, expected_sentiment):
    if predicted_sentiment is None or expected_sentiment is None:
        return None
    return predicted_sentiment != expected_sentiment

# Add derived inconsistency columns for both prompts
results_df["prompt_a_inconsistency_derived"] = results_df.apply(
    lambda row: derive_inconsistency(row["prompt_a_sentiment"], row["expected_sentiment"]),
    axis=1,
)
results_df["prompt_b_inconsistency_derived"] = results_df.apply(
    lambda row: derive_inconsistency(row["prompt_b_sentiment"], row["expected_sentiment"]),
    axis=1,
)


### Metric Functions

The following functions compute evaluation metrics:

- Sentiment accuracy: measures the agreement between predicted_sentiment (model output) and expected_sentiment (rating-derived reference).

- Inconsistency agreement: measures how often the model’s predicted inconsistency (predicted_inconsistency) matches the derived inconsistency label (derived_inconsistency).

In [18]:

# Compute sentiment accuracy: compares predicted_sentiment with expected_sentiment (rating-derived reference)
def compute_sentiment_accuracy(df, pred_col, gold_col="expected_sentiment"):
    valid = df[[pred_col, gold_col]].dropna()
    if len(valid) == 0:
        return None
    return accuracy_score(valid[gold_col], valid[pred_col])

# Compute inconsistency agreement: compares predicted_inconsistency with derived_inconsistency
def compute_boolean_agreement(df, pred_col, ref_col):
    valid = df[[pred_col, ref_col]].dropna()
    if len(valid) == 0:
        return None
    return (valid[pred_col] == valid[ref_col]).mean()



### Metric Computation

In [19]:
# Evaluate sentiment accuracy by comparing model-predicted sentiment with rating-derived expected sentiment
baseline_acc = compute_sentiment_accuracy(results_df, "baseline_sentiment")
prompt_a_acc = compute_sentiment_accuracy(results_df, "prompt_a_sentiment")
prompt_b_acc = compute_sentiment_accuracy(results_df, "prompt_b_sentiment")

# Measure how often the baseline sentiment (inferred from star rating) disagrees with the expected sentiment
baseline_inconsistency_rate = results_df["baseline_inconsistency"].mean()

# Compute inconsistency agreement by comparing model-predicted inconsistency with derived_inconsistency
prompt_a_inconsistency_agreement = compute_boolean_agreement(
    results_df, "prompt_a_inconsistency", "prompt_a_inconsistency_derived"
)
prompt_b_inconsistency_agreement = compute_boolean_agreement(
    results_df, "prompt_b_inconsistency", "prompt_b_inconsistency_derived"
)

# Measure how often model output fails JSON parsing
prompt_a_parse_error_rate = results_df["prompt_a_parse_error"].mean()
prompt_b_parse_error_rate = results_df["prompt_b_parse_error"].mean()

# Compile metrics into a table
metrics_table = pd.DataFrame([
    {"Metric": "Baseline Sentiment Accuracy", "Score": baseline_acc},
    {"Metric": "Prompt A Sentiment Accuracy", "Score": prompt_a_acc},
    {"Metric": "Prompt B Sentiment Accuracy", "Score": prompt_b_acc},
    {"Metric": "Prompt A Inconsistency Accuracy", "Score": prompt_a_inconsistency_agreement},
    {"Metric": "Prompt B Inconsistency Accuracy", "Score": prompt_b_inconsistency_agreement},
    {"Metric": "Prompt A Parse Error Rate", "Score": prompt_a_parse_error_rate},
    {"Metric": "Prompt B Parse Error Rate", "Score": prompt_b_parse_error_rate},
    {"Metric": "Baseline Inconsistency Rate", "Score": baseline_inconsistency_rate},
])

print(metrics_table)

                            Metric  Score
0      Baseline Sentiment Accuracy   0.54
1      Prompt A Sentiment Accuracy   0.91
2      Prompt B Sentiment Accuracy   0.97
3  Prompt A Inconsistency Accuracy   0.70
4  Prompt B Inconsistency Accuracy   1.00
5        Prompt A Parse Error Rate   0.00
6        Prompt B Parse Error Rate   0.00
7      Baseline Inconsistency Rate   0.46


Based on the quantitative evaluation results, Prompt B demonstrates superior performance across key metrics. Therefore, subsequent analysis focuses on Prompt B to further examine review quality.

In [20]:
# Analyze alignment between model-assessed review quality and helpfulness-based quality proxy
quality_alignment_table = pd.crosstab(
    results_df["prompt_b_review_quality"],
    results_df["quality_proxy"],
    margins=True,
)
quality_alignment_table


quality_proxy,high,low,medium,unknown,All
prompt_b_review_quality,,,,,
high,10,1,17,27,55
medium,12,4,12,17,45
All,22,5,29,44,100


## 8. LLM-as-Judge Evaluation

A second LLM is used as an evaluator to assess the quality of Prompt B outputs on a small subsample.

The judge scores:
- sentiment correctness
- inconsistency correctness
- reasoning quality

In [21]:
# Define the prompt template for the LLM judge to evaluate the model's predictions
JUDGE_PROMPT_TEMPLATE = """
You are a strict evaluator of LLM outputs for review analysis.

Evaluate the prediction using only the review text and rating below.

Review text:
{review_text}

Star rating:
{score}

Predicted sentiment:
{predicted_sentiment}

Predicted inconsistency:
{predicted_inconsistency}

Predicted reasoning:
{predicted_reasoning}

Scoring rules:
1. sentiment_correct: 1 if the predicted sentiment is justified by the review text, otherwise 0.
2. inconsistency_correct: 1 if the inconsistency decision is logically valid given the rating and review text, otherwise 0.
3. reasoning_quality: integer from 1 to 5, where 1 is weak or unsupported and 5 is clear, grounded, and concise.

Return JSON only:
{{
  "sentiment_correct": 0,
  "inconsistency_correct": 0,
  "reasoning_quality": 1,
  "judge_reasoning": "short explanation"
}}
""".strip()

# Function to evaluate model predictions with an LLM judge, including retries and error handling
def evaluate_with_llm_judge(row, model_name=JUDGE_MODEL_NAME, max_retries=2):
    judge_input = JUDGE_PROMPT_TEMPLATE.format(
        review_text=row["review_text"],
        score=row["Score"],
        predicted_sentiment=row["prompt_b_sentiment"],
        predicted_inconsistency=row["prompt_b_inconsistency"],
        predicted_reasoning=row["prompt_b_reasoning"],
    )

    required_keys = [
        "sentiment_correct",
        "inconsistency_correct",
        "reasoning_quality",
        "judge_reasoning",
    ]
    last_error = None

    for attempt in range(max_retries + 1):
        try:
            response = client.responses.create(
                model=model_name,
                input=[
                    {"role": "system", "content": "You are a strict evaluator. Return JSON only."},
                    {"role": "user", "content": judge_input},
                ],
                temperature=0,
                max_output_tokens=120,
            )

            raw_output = get_response_text(response)
            parsed_output, parse_error = safe_json_parse(raw_output, required_keys=required_keys)

            return {
                "judge_raw_output": raw_output,
                "judge_parse_error": parse_error,
                "judge_api_error": False,
                "judge_api_error_message": None,
                "judge_sentiment_correct": parsed_output.get("sentiment_correct"),
                "judge_inconsistency_correct": parsed_output.get("inconsistency_correct"),
                "judge_reasoning_quality": parsed_output.get("reasoning_quality"),
                "judge_reasoning": parsed_output.get("judge_reasoning"),
            }

        except Exception as e:
            last_error = str(e)
            time.sleep(min(2 ** attempt, 4))

    return {
        "judge_raw_output": None,
        "judge_parse_error": True,
        "judge_api_error": True,
        "judge_api_error_message": last_error,
        "judge_sentiment_correct": None,
        "judge_inconsistency_correct": None,
        "judge_reasoning_quality": None,
        "judge_reasoning": None,
    }


In [22]:
JUDGE_SAMPLE_SIZE = min(40, len(results_df))
# Select a subset of valid model outputs for LLM-based evaluation, prioritizing cases without parse errors and with valid predictions
judge_candidates = results_df[
    (results_df["prompt_b_parse_error"] == False)
    & (results_df["prompt_b_sentiment"].notna())
].copy()

# Randomly sample up to the predefined size; if fewer candidates are available, use all
judge_sample = judge_candidates.sample(
    n=min(JUDGE_SAMPLE_SIZE, len(judge_candidates)),
    random_state=RANDOM_STATE,
) if len(judge_candidates) > 0 else judge_candidates.copy()

# Run the LLM-based judge on the selected sample 
judge_results = []
for _, row in judge_sample.iterrows():
    judge_results.append({"Id": row["Id"], **evaluate_with_llm_judge(row)})

judge_df = pd.DataFrame(judge_results)
judge_df.head()


,Id,judge_raw_output,judge_parse_error,judge_api_error,judge_api_error_message,judge_sentiment_correct,judge_inconsistency_correct,judge_reasoning_quality,judge_reasoning
0,515953,"{\n ""sentiment_correct"": 1,\n ""inconsistency...",False,False,None,1,1,4,The predicted sentiment is justified as the re...
1,108072,"{\n ""sentiment_correct"": 1,\n ""inconsistency...",False,False,None,1,1,4,The predicted sentiment is correct as the revi...
2,111222,"{\n ""sentiment_correct"": 1,\n ""inconsistency...",False,False,None,1,1,3,The predicted sentiment is justified as the re...
3,247799,"{\n ""sentiment_correct"": 0,\n ""inconsistency...",False,False,None,0,0,2,The review expresses dissatisfaction with the ...
4,543324,"{\n ""sentiment_correct"": 1,\n ""inconsistency...",False,False,None,1,1,3,The review expresses mixed feelings about the ...


In [23]:
judge_summary = pd.DataFrame([
    {"Metric": "Judge sentiment correctness mean", "Score": judge_df["judge_sentiment_correct"].dropna().mean() if not judge_df.empty else None},
    {"Metric": "Judge inconsistency correctness mean", "Score": judge_df["judge_inconsistency_correct"].dropna().mean() if not judge_df.empty else None},
    {"Metric": "Judge reasoning quality mean", "Score": judge_df["judge_reasoning_quality"].dropna().mean() if not judge_df.empty else None},
    {"Metric": "Judge parse error rate", "Score": judge_df["judge_parse_error"].mean() if not judge_df.empty else None},
])

judge_summary


,Metric,Score
0,Judge sentiment correctness mean,0.925
1,Judge inconsistency correctness mean,0.925
2,Judge reasoning quality mean,3.775
3,Judge parse error rate,0.000


## 9. Examples of Model Success and Failure

In addition to aggregate metrics, this notebook inspects:
- successful cases
- failure cases

This helps identify where the prompts perform well and where they remain limited.

In [24]:
# Display full review texts and reasoning without truncation
pd.set_option("display.max_colwidth", None)

success_cases = results_df[
    (results_df["prompt_b_parse_error"] == False)
    & (results_df["prompt_b_sentiment"] == results_df["expected_sentiment"])
].copy()

# Sort successful cases by helpfulness ratio to
success_cases = success_cases.sort_values(by="helpfulness_ratio", ascending=False)
success_cases[
    [
        "Id", "Score", "expected_sentiment", "prompt_b_sentiment",
        "prompt_b_review_quality", "helpfulness_ratio", "review_text", "prompt_b_reasoning"
    ]
].head()




,Id,Score,expected_sentiment,prompt_b_sentiment,prompt_b_review_quality,helpfulness_ratio,review_text,prompt_b_reasoning
8,9518,1,negative,negative,high,1.0,"Hardly a baking powder. After a few uses, this powder spoilt a cake. It has lost its raising power. It is still well within the expiry date and the container has been kept closed and in a dry place. Avoid this one. I plan to ask for a refund.",The review clearly expresses dissatisfaction with specific details about the product's failure.
30,92104,2,negative,negative,high,1.0,"Tasted like Hamburger Helper to me.... Meh, not impressed. I am a huge fan of Velveeta Shells and Cheese so I thought I would really enjoy this as well. However, it didn't taste any different than your basic Hamburger Helper to me. It took about the same amount of time and simply had a packet of noodles, packet of ""seasoning"" (which, unfortunately, mine got punctured so it was chunky, but it dissolved during simmering) and packet of Velveeta cheese sauce. I wasn't impressed at all. I'd probably purchase whichever was cheaper if I was choosing at the grocery store.","The review expresses clear dissatisfaction with specific details, aligning with the 2-star rating."
5,429787,1,negative,negative,medium,1.0,"not so much. My title and rating are only due to the fact that this product is not from the USA... and I didn't realize that when I ordered it. I felt betrayed because in theory, it's a great snack... just apples, right? My son likes the taste and I thought all was good... but, the apples are imported... so if you're concerned about the whole arsenic issue of imported apples, then you may want to find a different product.","The review expresses dissatisfaction with the product's origin, aligning with the 1-star rating, and provides some detail about concerns."
14,6030,1,negative,negative,high,1.0,Ridiculous price. The advertised regular price of $99.99 is ridiculous. 5 hour energy drinks are widely available for $2.00 each. Don't buy from sellers who inflate prices and charge ridiculous amounts. You can get a case of 12 for $20 right here on Amazon.,"The review clearly expresses dissatisfaction with pricing, providing specific comparisons and details."
16,534648,1,negative,negative,high,1.0,"Fraudulent claims. I did my own test on this, and for me at least, it spiked my blood sugar almost identically to regular pasta. That is with cooking it to the exact time on the box, with the exact tiny portion size. This is indeed a case of- if it tastes too good to be true it probably is.","The review provides specific details about the test and results, clearly expressing a negative experience."


In [25]:
# Display full review texts and reasoning without truncation
pd.set_option("display.max_colwidth", None)

failure_cases = results_df[
    (results_df["prompt_b_parse_error"] == True)
    | (results_df["prompt_b_sentiment"] != results_df["expected_sentiment"])
].copy()

failure_cases[
    [
        "Id", "Score", "expected_sentiment", "prompt_b_sentiment",
        "prompt_b_inconsistency", "prompt_b_review_quality",
        "review_text", "prompt_b_reasoning", "prompt_b_raw_output"
    ]
].head()


,Id,Score,expected_sentiment,prompt_b_sentiment,prompt_b_inconsistency,prompt_b_review_quality,review_text,prompt_b_reasoning,prompt_b_raw_output
43,501565,3,neutral,negative,True,medium,needs somthing. This soda dosen't tast like rootbeer to me. It tast ok but it needs somthing. I will not order this flaver.,"The review expresses dissatisfaction with the soda, conflicting with the neutral 3-star rating.","{\n ""sentiment"": ""negative"",\n ""inconsistency"": true,\n ""review_quality"": ""medium"",\n ""reasoning"": ""The review expresses dissatisfaction with the soda, conflicting with the neutral 3-star rating.""\n}"
45,264388,3,neutral,negative,True,medium,crispy greens peaches are better.... These peaches taste like they weren't ripe or full of flavor when they dried them. I have had the crispy greens brand freeze dried peaches and they are really good but the package is a little smaller and they cost a little more. I was hoping these would be the same but they aren't nearly as good. unfortunately i bought a LOT of them....oh well.....,"The review expresses disappointment with the product, conflicting with the neutral 3-star rating.","{\n ""sentiment"": ""negative"",\n ""inconsistency"": true,\n ""review_quality"": ""medium"",\n ""reasoning"": ""The review expresses disappointment with the product, conflicting with the neutral 3-star rating.""\n}"
46,17161,3,neutral,negative,True,high,Not the Same. I'd eagerly awaited for delivery of this vanilla candy that I vividly remembered from childhood and was disappointed with this ersatz version of Bonomo's Turkish Taffy. It's not as rich and creamy in the mouth and doesn't have the real vanilla flavor that I recall in the original. I would bet that the ingredients are not the same; for one thing soy is now used.,"The review expresses disappointment and criticism, which contradicts the neutral sentiment typically associated with a 3-star rating.","{\n ""sentiment"": ""negative"",\n ""inconsistency"": true,\n ""review_quality"": ""high"",\n ""reasoning"": ""The review expresses disappointment and criticism, which contradicts the neutral sentiment typically associated with a 3-star rating.""\n}"


## 10. Final Summary

In [27]:
print("FINAL SUMMARY")
print("=" * 60)

print("Sample size:", len(results_df))
print("Baseline Sentiment Accuracy:", baseline_acc)
print("Prompt A Sentiment Accuracy:", prompt_a_acc)
print("Prompt B Sentiment Accuracy:", prompt_b_acc)
print("Prompt A Inconsistency Accuracy:", prompt_a_inconsistency_agreement)
print("Prompt B Inconsistency Accuracy:", prompt_b_inconsistency_agreement)
print("Prompt A Parse Error Rate:", prompt_a_parse_error_rate)
print("Prompt B Parse Error Rate:", prompt_b_parse_error_rate)

print("\nQuality alignment table:")
print(quality_alignment_table)

print("\nJudge summary:")
print(judge_summary)

FINAL SUMMARY
Sample size: 100
Baseline Sentiment Accuracy: 0.54
Prompt A Sentiment Accuracy: 0.91
Prompt B Sentiment Accuracy: 0.97
Prompt A Inconsistency Accuracy: 0.7
Prompt B Inconsistency Accuracy: 1.0
Prompt A Parse Error Rate: 0.0
Prompt B Parse Error Rate: 0.0

Quality alignment table:
quality_proxy            high  low  medium  unknown  All
prompt_b_review_quality                                 
high                       10    1      17       27   55
medium                     12    4      12       17   45
All                        22    5      29       44  100

Judge summary:
                                 Metric  Score
0      Judge sentiment correctness mean  0.925
1  Judge inconsistency correctness mean  0.925
2          Judge reasoning quality mean  3.775
3                Judge parse error rate  0.000
